# CWRU Bearing Dataset — Data Exploration

This notebook explores the raw vibration signals, generated spectrograms, and extracted features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from pathlib import Path
import yaml

with open('configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

print('Config loaded successfully')

## 1. Raw Signal Exploration
Load and visualize vibration signals from each .mat file.

In [ ]:
raw_dir = Path(cfg['paths']['raw_data'])
mat_files = sorted(raw_dir.glob('*.mat'))
print(f'Found {len(mat_files)} .mat files:')
for f in mat_files:
    print(f'  {f.name}')

In [ ]:
# Load and plot one signal from each fault type
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, mat_file in enumerate(mat_files):
    data = loadmat(str(mat_file))
    # Find the DE_time key
    de_key = [k for k in data.keys() if 'DE_time' in k]
    if de_key:
        signal = data[de_key[0]].flatten()
        axes[idx].plot(signal[:5000], linewidth=0.5)
        axes[idx].set_title(f'{mat_file.stem} ({len(signal)} samples)')
        axes[idx].set_xlabel('Sample')
        axes[idx].set_ylabel('Amplitude')
    else:
        print(f'No DE_time key in {mat_file.name}: {[k for k in data.keys() if not k.startswith("_")]}')

plt.tight_layout()
plt.show()

## 2. Feature Dataset Exploration

In [ ]:
df = pd.read_csv(cfg['paths']['feature_csv'])
print(f'Shape: {df.shape}')
print(f'\nFault classes: {df["fault"].nunique()}')
print(df['fault'].value_counts())
df.head()

In [ ]:
# Feature distributions by fault class
feature_cols = [c for c in df.columns if c != 'fault']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for idx, col in enumerate(feature_cols[:9]):
    ax = axes[idx // 3, idx % 3]
    for fault_class in df['fault'].unique():
        subset = df[df['fault'] == fault_class]
        ax.hist(subset[col], alpha=0.5, label=fault_class, bins=30)
    ax.set_title(col)
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()

## 3. Generated Spectrograms
Visualize spectrograms generated by the preprocessing pipeline.

First run: `python -m src.data_preprocessing.generate_spectrograms`

In [ ]:
from PIL import Image

spec_dir = Path(cfg['paths']['spectrograms'])
if spec_dir.exists():
    class_dirs = sorted([d for d in spec_dir.iterdir() if d.is_dir()])
    print(f'Spectrogram classes: {len(class_dirs)}')
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for idx, class_dir in enumerate(class_dirs[:10]):
        images = sorted(class_dir.glob('*.png'))
        count = len(images)
        if images:
            img = Image.open(images[0])
            axes[idx].imshow(img)
            axes[idx].set_title(f'{class_dir.name}\n({count} images)')
            axes[idx].axis('off')
    
    plt.suptitle('Sample Spectrograms by Fault Class', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f'Spectrogram directory not found at {spec_dir}')
    print('Run: python -m src.data_preprocessing.generate_spectrograms')

## 4. NPZ Pre-processed Data

In [ ]:
npz_path = cfg['paths']['npz_data']
data = np.load(npz_path)
print('Keys:', list(data.keys()))
for key in data.keys():
    print(f'  {key}: shape={data[key].shape}, dtype={data[key].dtype}')